In [0]:
from datetime import date

SNAPSHOT = date.today().isoformat()   # e.g. "2026-10-22"
BASE = f"/Volumes/launch_reliability/bronze/raw_files/{SNAPSHOT}"

GCAT = "https://www.planet4589.org/space/gcat/tsv"
SOURCES = {
    "launch":  f"{GCAT}/launch/launch.tsv",
    "lv":      f"{GCAT}/tables/lv.tsv",
    "family":  f"{GCAT}/tables/family.tsv",
    "lvs":     f"{GCAT}/tables/lvs.tsv",
    "stages":  f"{GCAT}/tables/stages.tsv",
    "engines": f"{GCAT}/tables/engines.tsv",
    "orgs":    f"{GCAT}/tables/orgs.tsv",
    "sites":   f"{GCAT}/tables/sites.tsv",
}

In [0]:
import os, urllib.request

os.makedirs(BASE, exist_ok=True)

for name, url in SOURCES.items():
    path = f"{BASE}/{name}.tsv"
    urllib.request.urlretrieve(url, path)
    print(f"downloaded {name}: {os.path.getsize(path):,} bytes")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

def read_gcat_header(path):
    with open(path) as f:
        header = f.readline().rstrip("\n")
        updated = f.readline().rstrip("\n")
    assert header.startswith("#"), f"Unexpected header in {path}"
    return header.lstrip("#").split("\t"), updated.lstrip("#").strip()

def load_bronze(name):
    path = f"{BASE}/{name}.tsv"
    cols, updated = read_gcat_header(path)
    schema = StructType([StructField(c, StringType(), True) for c in cols])

    df = (spark.read
          .option("sep", "\t")
          .option("comment", "#")
          .option("header", "false")
          .option("quote", "\u0000")
          .schema(schema)
          .csv(path)
          .withColumn("_source_file", F.lit(path))
          .withColumn("_gcat_updated", F.lit(updated))
          .withColumn("_snapshot_date", F.lit(SNAPSHOT))
          .withColumn("_ingested_at", F.current_timestamp()))

    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"launch_reliability.bronze.{name}"))

    print(f"{name}: {df.count():,} rows, {len(cols)} columns | {updated}")

In [0]:
for name in SOURCES:
    load_bronze(name)

In [0]:
%sql
-- column names and types
DESCRIBE launch_reliability.bronze.launch;

-- a sample of actual rows
SELECT * FROM launch_reliability.bronze.launch LIMIT 20;